# Re-run: planted defects and which checks carry them (RQ3)

This notebook recomputes the point estimates of Table 4 (each format's change when a false or key-terms
sentence replaces a length-matched neutral sentence on the same answer) and Table 5 (the checklist's
change split into its error checks and its presence checks) from the per-case file published with
release `checklist-grading-2026`. It needs Python 3.9+ with numpy and `per-case-model-outputs.jsonl`
beside this notebook (or in `RELEASE_DIR`).

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "checklist-grading-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import json
import numpy as np

recs = [json.loads(l) for l in open(find_release_file(), encoding="utf-8")]
G = {(r["id"], r["form"], r["model"], r["arm"], r["rep"]): r for r in recs if r["record"] == "grade" and r["corpus"] == "mohler"}
items = {r["id"]: r for r in recs if r["record"] == "item" and r["corpus"] == "mohler"}
Q = {r["q"]: r for r in recs if r["record"] == "question" and r["corpus"] == "mohler"}
MODELS, ARMS, REPS = ("gemini", "gpt"), ("likert", "boolean"), (0, 1)

def answers(kind):
    return [i for i, it in sorted(items.items()) if kind in it and it[kind]["ok"] and it[kind]["manip_pass"]]

def score(i, form, m, a):
    return np.mean([G[i, form, m, a, k]["score"] for k in REPS])

In [3]:
PRINTED4 = {"contra": (120, "-0.473", "-0.172", "+0.302"), "stuff": (112, "+0.273", "+0.348", "+0.075")}
for kind, want in PRINTED4.items():
    ids = answers(kind)
    D = {a: np.mean([np.mean([score(i, f"{kind}-treat", m, a) - score(i, f"{kind}-neutral", m, a) for i in ids])
                     for m in MODELS]) for a in ARMS}
    got = (len(ids), f"{D['likert']:+.3f}", f"{D['boolean']:+.3f}", f"{D['boolean'] - D['likert']:+.3f}")
    print(kind, got)
    assert got == want, (kind, got)
print("Table 4's point estimates reproduce.")

contra (120, '-0.473', '-0.172', '+0.302')
stuff (112, '+0.273', '+0.348', '+0.075')
Table 4's point estimates reproduce.


In [4]:
def split(kind, subset=None):
    rows = []
    for i in answers(kind):
        avoid = np.array(Q[items[i]["q"]]["avoid_by_check"], bool)
        if subset is not None and avoid.any() != subset:
            continue
        per_m = []
        for m in MODELS:
            y = {f: np.mean([G[i, f"{kind}-{f}", m, "boolean", k]["checks"] for k in REPS], axis=0) for f in ("treat", "neutral")}
            d = y["treat"] - y["neutral"]
            per_m.append((d.sum() / len(d), d[avoid].sum() / len(d), d[~avoid].sum() / len(d)))
        rows.append(np.mean(per_m, axis=0))
    rows = np.array(rows)
    return len(rows), rows.mean(0)

n, (D, A, P) = split("contra")
print(f"false sentence, all: n={n} D {D:+.3f} A {A:+.3f} P {P:+.3f}")
assert (n, f"{D:+.3f}", f"{A:+.3f}", f"{P:+.3f}") == (120, "-0.172", "-0.109", "-0.063")
n, (D, A, P) = split("contra", True)
print(f"false sentence, with error checks: n={n} D {D:+.3f} E {A:+.3f} share {A / D:.3f}")
assert (n, f"{D:+.3f}", f"{A:+.3f}", f"{A / D:.3f}") == (77, "-0.224", "-0.170", "0.760")
n, (D, A, P) = split("contra", False)
print(f"false sentence, no error check: n={n} D {D:+.3f}")
assert (n, f"{D:+.3f}") == (43, "-0.079")
n, (D, A, P) = split("stuff")
print(f"key terms, all: n={n} D {D:+.3f} A {A:+.3f} P {P:+.3f}")
assert (n, f"{D:+.3f}", f"{A:+.3f}", f"{P:+.3f}") == (112, "+0.348", "+0.003", "+0.346")
print("Table 5's point estimates reproduce.")

false sentence, all: n=120 D -0.172 A -0.109 P -0.063
false sentence, with error checks: n=77 D -0.224 E -0.170 share 0.760
false sentence, no error check: n=43 D -0.079
key terms, all: n=112 D +0.348 A +0.003 P +0.346
Table 5's point estimates reproduce.
